# Exploratory Data Analysis â€” Event-Log Dataset

> **Status: COMPLETE -- Sections 4-18 measured (Stages 4-7).** All findings are measured; no causal claims.

Conventions: descriptive variable names, one logical step per cell, named constants from `src.data_processing` (no magic numbers), every chart answers a ranked question.

In [ ]:
# 0. Reproducible setup (imports + style + light file inventory only)
import sys
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR / ".." / "src").exists() else NOTEBOOK_DIR
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_processing import (
    RANDOM_STATE,
    SESSION_GAP_MINUTES_CANDIDATES,
    SESSION_GAP_MINUTES_DEFAULT,
    list_raw_files,
)
from src.visualization import apply_report_style

np.random.seed(RANDOM_STATE)
apply_report_style()

print(f"RANDOM_STATE={RANDOM_STATE}, SESSION_GAP_DEFAULT={SESSION_GAP_MINUTES_DEFAULT}, CANDIDATES={SESSION_GAP_MINUTES_CANDIDATES}")
for existing_raw_file in list_raw_files(REPO_ROOT):
    print(f"found: {existing_raw_file.relative_to(REPO_ROOT)}")


## 1. Project Overview

Three-table e-commerce interaction log (2,756,101 events, 20,275,902 EAV property rows, 1,669-node category tree, May-Sep 2015) with no dictionary, no target, and no session id. This notebook earns every modeling decision from measurement: Sec 4 inventories, Sec 5/10/11 contain quality and missingness, Sec 6-8 build the behavioral evidence, Sec 9 selects the session-buy label, Sec 12-13 verify relationships with rank methods and pre-registered tests, Sec 14 proposes features without building them, and Sec 15-18 convert the evidence into a readiness verdict, findings, and recommendations. Associations are never read as causes; nothing is silently cleaned.


## 2. Analytical Objective

Determine whether this clickstream can support a production-grade purchase-intent / recommendation model, and what must be proven first. Stakeholders: ranking (candidate features, negative sampling), merchandising (funnel breaks, stockout coincidence), platform (as-of join spec, session rule, chunked pipeline), trust (bot/artifact quantification). Success = a defensible label, a leak-free feature plan, and enforceable readiness gates -- delivered in Sec 9 and Sec 15.


## 3. Analytical Questions

Ranked in Stage 2 (P0 = critical, P1 = important, P2 = opportunistic). Q1 funnel/learnability -> Sec 7, 9. Q2 sessionization -> Sec 7 (30-min default, gap-robust). Q3 as-of vs naive join -> Sec 7 (25.5% disagreement). Q4 availability -> Sec 7, 8 (9x lift, survives popularity split). Q5 hierarchy -> Sec 7 (45x smoothed spread). Q6 concentration -> Sec 6-8 (head converts better). Q7 duplicates/artifacts -> Sec 5, 10 (quarantine, retain). Q8 property coverage -> Sec 6 (head encodable, tail hashed). Q9 price transforms -> Sec 6, 12 (log/quantile candidates). Q10 baskets/missingness bias -> Sec 6, 11 (event grain stays; missingness informative).


## 4. Dataset Overview (completed Stage 4 — measured, not printed raw)

**Structure (fact).** Three linked tables, not one flat file: `events` (2,756,101 x 5, ~121.8 MB in memory), `category_tree` (1,669 x 2, ~26.8 KB), `item_properties` EAV long format (20,275,902 rows total: 10,999,999 + 9,275,903 across two parts, 4 cols). Raw dtypes: events `timestamp int64 / visitorid int64 / event str / itemid int64 / transactionid float64`; tree `categoryid int64 / parentid float64` (float only from NaNs); properties `timestamp int64 / itemid int64 / property str / value str`.

**Time (fact).** All `timestamp`s are epoch-ms: events 1430622004384 (2015-05-03) to 1442545187788 (2015-09-18), ~4.5 months / 20 full weeks + 2 partial edge weeks. Property snapshots span 1431226800000-1442113200000 (same window) — hence the as-of join requirement (Q3).

**Cardinality (fact).** events: `timestamp 2,750,455 distinct / visitorid 1,407,580 / itemid 235,061 / transactionid 17,672 / event 3`. Tree: `categoryid` unique (1,669), `parentid` 362 distinct + 25 NaN roots. Properties: 1,104 distinct `property` codes over 20,275,902 rows; top-5 codes hold 38.0% of rows, top-20 hold 67.9%, top-50 hold 80.6% (long sparse tail; several codes occur once). Property universe covers 417,053 items per part vs. 235,061 interacted items — most catalog items are never interacted with.

**Missingness (fact).** `transactionid` missing 2,733,644/2,756,101 (99.19%) — structural: non-null iff `event == 'transaction'` (22,457 rows, 0 violations either way). `parentid` missing 25/1,669 (1.5%, the roots). Zero missing elsewhere.

**Duplicates (fact).** 460 fully-duplicated event rows (0.017%), concentrated in `addtocart` repeats at identical ms timestamps (double-logging hypothesis, Q7). Tree: 0 dups. Properties: 0 dups in 500k sample (full-file check deferred to Stage 6 for cost reasons).

**Interpretation.** Do not model `timestamp` ints as magnitudes; do not pivot properties wide (sparse x huge); do not treat `transactionid` absence as imputable. Head-vs-tail structure (see Univariate) and the EAV sparsity pattern are the two facts that shape every downstream choice.


In [ ]:
# 4a. Dataset Overview — reproducible inventory from measured summaries (no heavy parse)
import json
from pathlib import Path
from IPython.display import Image, display

overview_events = json.loads(Path("../reports/stage4_events_cat.json").read_text())
print("events shape:", overview_events["events_shape"], f"({overview_events['events_memory_MB']} MB)")
print("dtypes:", overview_events["events_dtypes"])
print("missing %:", overview_events["events_missing_pct"])
print("nunique:", overview_events["events_nunique"])
print("event mix:", overview_events["event_counts"], overview_events["event_pct"])
print("category_tree shape:", overview_events["cat_shape"], "roots:", overview_events["cat_roots"])
display(Image(filename="../reports/figures/fig_event_mix.png"))
display(Image(filename="../reports/figures/fig_weekly_volume.png"))


## 5. Data Quality Assessment (completed Stage 4 — documented, nothing silently cleaned)

**DQ1 — `transactionid` 99.19% missing is structural, not imputable.** Observation: 2,733,644 missing; cross-tab shows `notna() ⟺ event == 'transaction'` exactly (addtocart 69,332 all-null, view 2,664,312 all-null, transaction 22,457 all-non-null). Evidence: full-file crosstab, 0 counter-examples. Impact: any imputation or `transactionid`-as-feature leaks the candidate label perfectly. Action: keep as basket-grouping key only; ban as feature (Q1/Q3 gate).

**DQ2 — 460 exact-duplicate events.** Observation: 0.017% fully-duplicated rows, spot-checked as same visitor+item+ms-timestamp `addtocart` pairs. Evidence: `duplicated().sum() == 460`. Impact: negligible for global rates, but a 0.66% uplift risk inside the small `addtocart` class if double-logging. Action: quarantine-and-sensitivity (rates with/without dups) in Stage 6; no deletion now.

**DQ3 — `visitorid == 0` (3 rows).** Observation: 3 `view` rows carry visitor 0 (min valid id is otherwise 1). Evidence: direct filter. Impact: nil statistically; breaks per-visitor joins if treated as a real user. Action: quarantine from visitor-level features, retain in global counts with a note (Q7).

**DQ4 — Type traps.** Observation: `transactionid float64` and `parentid float64` exist only because of NaNs; `value` is `str` dtype over ≥4 semantic formats (bare token 45%, multi-token 32%, `n`-single 11%, binary 7%, mixed 5% in 400k sample; median length 7, p99 248, max 437). Evidence: dtype table + format classifier + length descriptives. Impact: naive `astype(int/float)` or `.describe()` on `value` corrupts or crashes. Action: nullable dtypes on load; per-`property` parsing functions (Q8); never describe `value` pooled.

**DQ5 — Opaque property codes + multi-token values.** Observation: 1,104 codes; only `available` (binary 0/1: 57.8/42.2 in sample) and `categoryid` (845 distinct in sample) are face-interpretable; e.g. code `283` emits 40+-token rows; `790` single-`n` values in sample are all parseable numerics but range 0–17M (median 46,440). Evidence: top-30 coverage file + format-by-property table. Impact: semantic use of any other code is currently a guess. Action: frequency/format-gated allow-list; `283`-style lists need tokenizer design, not scalar casts.

**DQ6 — Temporal leakage surface (flagged, quantified in Stage 5).** Observation: property snapshots overlap the event window, so latest-value joins mix future state into past events. Evidence: overlapping ms ranges (see §4). Impact: inflated offline metrics + training-serving skew. Action: as-of (`property_time <= event_time`) joins only; Stage 5 measures naive-vs-as-of disagreement (Q3). No merge performed in this stage.

**DQ7 — No invalid numerics found.** Observation: zero `timestamp <= 0`, zero `itemid <= 0`; no impossible dates outside May–Sep 2015. `event` has exactly 3 consistent spellings. Action: none; range assertions go into `tests/` as pipeline guards.


In [ ]:
# 5a. Data Quality — re-derive the three structural checks (light: events only, properties via sample)
import pandas as pd

dq_events = pd.read_csv("../data/raw/events.csv")  # full row: subset dup counts inflate (1.3M on 3 cols vs 460 full-row)
print("transactionid missing %:", round(dq_events["transactionid"].isna().mean() * 100, 4))
print(pd.crosstab(dq_events["event"], dq_events["transactionid"].notna()).to_string())
print("fully-duplicated rows:", int(dq_events.duplicated().sum()))
print("visitorid==0 rows:", int((dq_events["visitorid"] == 0).sum()))
print("event spellings:", sorted(dq_events["event"].unique()))


## 6. Univariate Analysis (completed Stage 4 — each chart answers a ranked question)

**Target-candidate distribution (Q1).** `view` 2,664,312 (96.67%), `addtocart` 69,332 (2.52%), `transaction` 22,457 (0.81%) — see `fig_event_mix.png` (log scale; linear scale would erase the minority classes). Weekly volume (`fig_weekly_volume.png`, 20 weeks) is flat except partial edge weeks (w/c 2015-04-27: 13,304 views; w/c 2015-09-14: 62,289 views vs. ~115-170k mid-window): no mid-window outage, edges must be excluded from rate comparisons. Implication: accuracy is a vanity metric here; Stage 5 frames conditional funnel rates + PR-AUC/recall@k.

**Visitor behaviour (Q6).** Events-per-visitor: mean 1.96, median 1, p90 3, p99 13, max 7,757; skew 269.5, kurtosis ~124k. 71.15% of 1.41M visitors appear once. Concentration is mild: top 10% of visitors hold 42.3% of events. Implication: single-event visitors dominate — history features will be missing-by-construction for most users (cold-start is the norm, not an edge case).

**Item popularity (Q6).** Events-per-item: mean 11.73, median 3, p90 25, p99 143, max 3,412 (item 187946); skew 19.3. Concentration is steep: top 1% of items take 22.9% of events, top 10% take 64.0%, top 20% take 78.7% (`fig_concentration.png`). Implication: popularity baselines will look strong; evaluation must include tail slices or the model just memorizes the head.

**Inter-event gaps (Q2 input, not a session claim).** Within-visitor gaps (1.35M multi-event gaps): median 2.3 min, p90 ~73 h, mean 3,571 min (mean >> median = heavy tail); 73.7% ≤ 30 min (`reports/stage6_gaps_extra.json`). Return periodicity is visible as same-day/next-day spikes: 2.25% of gaps fall in 20-28h and 0.56% in 6.5-7.5d (peaks near 1,428 and 9,968 min) -- evidence for 1d/7d recency harmonics in feature F7, not just a session threshold., 76.3% ≤ 60 min (`fig_gap_hist.png`, log-x). Implication: a 30-min gap rule keeps ~3/4 of continuations in-session; Stage 5 runs the 15/30/60-min sensitivity before fixing the rule. No sessions are assigned in this stage.

**Basket size (Q10 context).** 17,672 baskets from 22,457 transaction events: mean 1.27, median 1, p90 2, p99 5, max 31; only 15.33% of baskets are multi-item; skew 10.1. Implication: basket-level modeling is thin — event-level framing stays primary.

**Property coverage (Q8).** Top codes by rows of 20.28M: `888` 3.00M (14.8%), `790` 1.79M, `available` 1.50M, `categoryid` 0.79M, `6` 0.63M (`fig_property_coverage.png`); top-20 cover 67.9%, tail of ~1,050 codes covers the rest. `available` is 57.8% `0` / 42.2% `1` (400k sample) — near-balanced, encodable as-is. `categoryid` shows 845 distinct values in-sample. Implication: encode the head, hash-or-drop the tail; multi-token codes (`283`-style, max length 437) need tokenizers, not scalars.

**Numeric shape notes (transform triage, Q9 conditional).** `value` length: mean 17.7, median 7, p99 248, skew 5.7. Code-`790` single-`n` numerics (n=35,417 in sample): median 46,440, p99 ~1.0M, max 17.2M, skew 19.8 — log/quantile transform *candidate if* Stage 5 shows it is item-varying and as-of available; no transform applied now. Visitor/item/basket counts all have kurtosis >> 3: rank/quantile methods preferred over raw-scale parametric tests.

**What was NOT done (by design).** No bivariate/multivariate relationships, no target conditioning, no session assignment, no property↔event joins, no outlier removal — those belong to Stages 5–6.


In [ ]:
# 6a. Univariate — render measured distributions (figures precomputed; stats from JSON)
import json
from pathlib import Path
from IPython.display import Image, display

univariate_stats = json.loads(Path("../reports/stage4_univariate.json").read_text())
for variate_name, descriptives in univariate_stats.items():
    if isinstance(descriptives, dict) and "median" in descriptives:
        print(f"{variate_name}: median={descriptives['median']}, p90={descriptives['q90']}, "
              f"max={descriptives['max']}, skew={descriptives['skew']}, kurt={descriptives['kurt']}")
    else:
        print(f"{variate_name}: {descriptives}")
for figure_name in ["fig_concentration.png", "fig_gap_hist.png",
                    "fig_property_coverage.png", "fig_value_length.png"]:
    display(Image(filename=f"../reports/figures/{figure_name}"))


## 7. Bivariate Analysis (completed Stage 5 — associational only, no causal claims)

**7.1 Session-gap sensitivity (Q2).** Observation: session counts barely move across 15/30/60-min gaps — 1,809,161 → 1,761,675 → 1,726,714 (a 4.6% band); single-event share 79.08% → 78.27% → 77.64%. Evidence: `reports/stage5_sessions.json` + `fig_session_sensitivity.png`. Interpretation: the 73.7%-under-30-min gap mass (Stage 4) makes the partition gap-robust. Why it matters: the 30-min default (now implemented as `src.data_processing.sessionize_events`) is a safe, documented choice rather than a tuned parameter; downstream session features will not hinge on the threshold.

**7.2 As-of availability × event type (Q3/Q4).** Observation: in a 200k-event sample with point-in-time join, in-stock events carry 4.13% carts / 1.36% transactions vs 0.44% / 0.15% out-of-stock — roughly a 9x gap at both funnel stages; as-of coverage is 80.8%. Evidence: `reports/stage5_asof.json` + `fig_asof_available.png`. Interpretation: availability state is the strongest single associate of conversion found so far — but popularity is an unadjusted confounder (popular items are restocked and bought), so this is a feature candidate, not an effect estimate. Why it matters: justifies an as-of `available` flag as a first-class modeling feature *and* mandates the as-of pipeline — the naive latest-value join misstates state for 25.5% of sampled events (Q3 leakage quantified: 90.8% naive vs 80.8% as-of coverage).

**7.3 Session length × outcome (Q1 framing + leakage flag).** Observation: events/session mean 9.34 (median 4) for buy sessions, 4.70 (3) for cart-no-buy, 1.44 (1) for view-only. Evidence: `reports/stage5_sesslen.json` + `fig_session_length_outcome.png`. Interpretation: outcome separates cleanly by session depth. Why it matters: supports session-grain labels (§9) — but total session length is knowable only post-hoc, so it is banned as an intra-session feature (leakage).

**7.4 Item popularity × conversion (Q6).** Observation: across 8 view-deciles (ties collapsed 10→8), carts/1000 views rise 9.3 → 30.9 and tx/1000 views 0.72 → 10.31 monotonically; tx/100 carts rise 7.7 → 33.3. Evidence: `reports/stage5_deciles.csv` + `fig_popularity_conversion.png`. Interpretation: the head converts better at *both* stages, not just by volume. Why it matters: popularity-decile features will dominate naively — evaluation must include tail slices or the model memorizes the head.

**7.5 Root category × conversion (Q5).** Observation: 23 roots observed in events; EB-smoothed (m=500) tx rates span 1.78% (root 1224) to 0.04% (root 431) — a ~45x spread; tiny roots (e.g. n=2) shrink to the 0.81% global mean instead of showing 0%. Evidence: `reports/stage5_roots.csv` + `fig_category_conversion.png`. Interpretation: branch carries signal beyond item identity; smoothing prevents small-n whiplash. Caveat: categories use latest-known (not as-of) mapping — association only. Why it matters: backs hierarchical back-off encodings for cold-start items.

**7.6 Week × conversion (Q1 split validity).** Observation: weekly tx/1000 views holds 7.2–9.7 across all 20 full weeks (median line flat; edge partial weeks excluded). Evidence: `reports/stage5_weekly_tx.csv` + `fig_weekly_txrate.png`. Interpretation: no regime shift or mid-window outage. Why it matters: temporal train/validation splits are valid; no need for regime dummies.


In [ ]:
# 7a. Bivariate — re-render measured relationships (precomputed JSONs + src cross-check)
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

bivariate_sessions = json.loads(Path("../reports/stage5_sessions.json").read_text())["gaps"]
print(pd.DataFrame(bivariate_sessions).T.round(2).to_string())
bivariate_asof = json.loads(Path("../reports/stage5_asof.json").read_text())
print("as-of coverage %:", bivariate_asof["asof_coverage_pct"],
      "| naive-vs-asof disagreement %:", bivariate_asof["naive_vs_asof_disagreement_pct"])

# src cross-check on a small live sample (functions unit-tested in tests/test_data_processing.py)
from src.analysis import funnel_rates
spot_sample = pd.read_csv("../data/raw/events.csv", usecols=["event"], nrows=100000)
print("spot funnel/1000 views:", funnel_rates(spot_sample)["tx_per_1000_views"], "(full-file: 8.43)")

for bivariate_fig in ["fig_session_sensitivity.png", "fig_asof_available.png",
                      "fig_session_length_outcome.png", "fig_popularity_conversion.png",
                      "fig_category_conversion.png", "fig_weekly_txrate.png"]:
    display(Image(filename=f"../reports/figures/{bivariate_fig}"))


## 8. Multivariate Analysis (completed Stage 5 — every analysis has a reason; no pairplot)

**8.1 Availability lift × popularity half (Q4×Q6: is availability just a popularity proxy?).** Observation: the in-stock tx lift persists within both halves — head 0.17%→1.36%, tail 0.01%→1.23%; cart lift likewise (head 0.48%→4.10%, tail 0.11%→5.83%). Evidence: `reports/stage5_avail_pop.json` (median split at 3 views) + `fig_avail_pop.png`. Interpretation: availability is not reducible to popularity — the lift survives the split, so both deserve separate features. Caveat: the tail out-of-stock cell holds just 7,288 events with 1 transaction — its 0.01% is high-variance and must not be over-read. Why it matters: approves two independent feature families (as-of availability flag + popularity-decile priors) instead of one proxy.

**8.2 Item-metric redundancy (which derived columns survive?).** Observation: Spearman on 235k items — views↔visitors 0.976 (near-perfect redundancy), carts↔txs 0.681 (linked but distinct stages), views↔txs 0.326 (volume ≠ conversion). Evidence: `reports/stage5_corr_items.csv` + `fig_corr_items.png` (4 pre-registered derived metrics only — no blind pairplot). Interpretation: distinct-visitors adds nothing over view counts at item grain; carts and transactions must stay separate targets/stages. Why it matters: drop `visitors` from item-level feature sets (collinear); keep stage-separated labels; prefer rank methods (all inputs kurtosis >> 3).

**8.3 Session-depth confounding (why length is signal and leakage at once).** Observation: buy sessions average 9.3 events, yet 78% of all sessions are single-event and 71% of visitors appear once (§6). Evidence: §§7.3 + 7.1 jointly. Interpretation: length discriminates precisely because most sessions are too short to convert — conditioning features on full-session depth would select on the outcome. Why it matters: session features must be prefix-legal (computed from events so far); total length joins `transactionid` on the banned list.

**8.4 Segmentation preview for Stage 6.** The three splits that earned follow-up: (a) high- vs low-converting roots × availability (does stockout hurt more in fixed-supply branches?), (b) head vs tail × session depth (do tail conversions need longer sessions?), (c) early- vs late-window × weekly rate (flat now — re-check after any resampling). No further segmentation is run here to avoid dredging.


In [ ]:
# 8a. Multivariate — interaction + redundancy from measured artifacts (no new joins)
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

interaction = json.loads(Path("../reports/stage5_avail_pop.json").read_text())["conv_pct"]
print("tx% head out-of-stock:", interaction["head|avail=0|transaction"],
      "-> in-stock:", interaction["head|avail=1|transaction"])
print("tx% tail out-of-stock:", interaction["tail|avail=0|transaction"],
      "-> in-stock:", interaction["tail|avail=1|transaction"])
print(pd.read_csv("../reports/stage5_corr_items.csv", index_col=0).round(2).to_string())
display(Image(filename="../reports/figures/fig_avail_pop.png"))
display(Image(filename="../reports/figures/fig_corr_items.png"))


## 9. Target Variable Analysis (completed Stage 5 — a target is *chosen*, not found)

**Status of the target.** Fact: no label column exists. Decision (revisable): **session-level `buy` (session contains ≥1 transaction) as primary**, with event-level `transaction` and visitor-level ever-buyer as alternatives. Rationale: sessions are the decision unit (27.2% of cart sessions convert vs 32.4 tx/100 carts event-upper-bound), session ids are now deterministic (`sessionize_events`, gap-robust per §7.1), and session grain avoids multi-item-basket ambiguity (15.3% of baskets are multi-item).

**Class distribution (the imbalance reality).** Event: 0.815% tx. Session (30-min): 0.812% buy / 1.816% cart-no-buy / 97.372% view-only (14,297 / 31,992 / 1,715,386 of 1,761,675). Visitor: 0.833% buyers (11,719 of 1,407,580). Evidence: `reports/stage5_funnel.json` + `fig_funnel.png` — the ~0.8% rate survives all three grains, so it is structural, not an aggregation artifact. Implication: accuracy is meaningless; optimize PR-AUC / recall@fixed-precision; plan class weights or negative downsampling of view-only sessions; stratify splits by outcome *and* week.

**Target vs features (discriminative preview, associational).** As-of in-stock → 9x tx rate (§7.2, survives popularity split §8.1). Root branch → 45x smoothed spread (§7.5). Popularity decile → 14x tx/1000-views spread (§7.4). Session depth → separates outcomes but is post-hoc (§7.3/§8.3). None is causal; together they imply a learnable conditional structure far above the 0.8% base rate — the modeling question is legitimate.

**Target quality concerns.** (1) 460 duplicate `addtocart` rows inflate the cart stage ~0.7% — keep the quarantine policy. (2) 19.2% of sampled events lack as-of availability (item unseen before event time) — the feature needs an explicit unknown state, not imputation. (3) Latest-category mapping (not as-of) may misattribute branch for items that changed category — flag, don't fix, at this stage.

**What this means for downstream ML.** Recommended framing: session-buy classification with prefix-legal features only (as-of availability + unknown state, root back-off encoding, popularity-decile prior, within-session prefix dynamics), temporal split (last weeks held out; weekly rate is stationary per §7.6), session-aware grouping (never split events of one session across folds), PR-AUC primary with recall@precision operating points. Banned: `transactionid`, total session length, naive latest-value property joins. The funnel math also fixes sampling budgets: ~26 carts and ~8 transactions per 1000 views.


In [ ]:
# 9a. Target — funnel at three grains from the measured artifact (no refitting)
import json
from pathlib import Path
from IPython.display import Image, display

target_funnel = json.load(open("../reports/stage5_funnel.json"))
print("event-level per 1000 views:", target_funnel["event_level"])
print("session-level outcome %:", target_funnel["session_level"]["outcome_mix_pct"])
print("visitor-level outcome %:", target_funnel["visitor_level"]["outcome_mix_pct"])
print("cart sessions converting to buy %:",
      target_funnel["session_level"]["cart_sessions_converting_to_buy_pct"])
display(Image(filename="../reports/figures/fig_funnel.png"))
display(Image(filename="../reports/figures/fig_weekly_txrate.png"))


## 10. Outlier Investigation (completed Stage 6 — triaged, nothing removed)

Method note: blind fences fail here — the IQR upper fence flags 8.6% (events/visitor) to 22.1% (gaps) as outliers, and mean+3sd flags 0.1–2.4% (`fig_outlier_triage.png`). With skew 5–270 and kurtosis up to ~124k, fences describe the tail rather than detect errors. Dispositions below come from case inspection, not fence membership.

**O1 — Hyperactive visitors (max 7,757 events).** Observation: top-5 visitors span 31–135 days, touch 1,314–3,814 distinct items across 54–414 sessions, and convert at 2.0–11.5% — at or above the 0.8% base rate. Evidence: `reports/stage6_outliers.json` top_visitor_cases. Triage: genuine extremes (power shoppers; automated assistance cannot be ruled out but they buy, so exclusion would bias the target). Disposition: **retain**; use rank/decile representations so their mass does not dominate scale-sensitive models. Never delete for volume.

**O2 — Large baskets (max 31 items, all distinct).** Observation: the five largest baskets (21–31 items) contain 14–31 distinct items with top-item share 3–7% — broad orders, not quantity typos of one SKU. Evidence: top_basket_cases. Triage: rare but valid. Disposition: **retain**; log-transform basket-size features if used; cap only inside explicitly winsorized model variants with the cap reported.

**O3 — Marathon sessions (max 417 events, view-only over 389 items).** Observation: the longest sessions split across outcomes (view-only 417/330/275 and buy 320/310) with 157–389 distinct items each. Evidence: top_session_cases. Triage: crawler-like yet plausibly comparison shopping; outcome-mixed, so no label rule separates them. Disposition: **retain + monitor** (rate-limit flag upstream); session length stays banned as a feature (§8.3) which already neutralizes their leverage.

**O4 — Price-like `790` tail (max 17.2M vs median 46,440, skew 19.8).** Observation: single-`n` values span 0 to 17,176,800 with IQR-upper 291,120 and 10.3% above it. Evidence: prop790 block. Triage: cannot separate unit errors (cents vs units) from genuine luxury items without the dictionary — an explicit unknown. Disposition: **transform, don't trim** — log1p/quantile mapping if the feature earns inclusion; no deletion, no winsor cap chosen at this stage.

**O5 — Gaps (max ~196k min ≈ 136 days).** Observation: within-visitor gaps reach the full window length; 22% exceed the IQR fence of 101 min. Triage: genuine return intervals, defining session boundaries. Disposition: **retain**; gaps above the session threshold become new-session starts by construction.


In [ ]:
# 10a. Outliers — fence comparison + top-case triage (measured artifact)
import json
import pandas as pd

outlier_bounds = json.load(open("../reports/stage6_outliers.json"))
print(pd.DataFrame({k: v for k, v in outlier_bounds.items()
                    if isinstance(v, dict) and "above_iqr_pct" in v}).T
      [["median", "iqr_upper", "above_iqr_pct", "above_mean_plus_3sd_pct", "max"]].round(2).to_string())
print("top-visitor tx%:", [c["mix"].get("transaction") for c in outlier_bounds["top_visitor_cases"]])
print("top-basket distinct items:", [c["distinct_items"] for c in outlier_bounds["top_basket_cases"]])
from IPython.display import Image, display
display(Image(filename="../reports/figures/fig_outlier_triage.png"))


## 11. Missing Data Analysis (completed Stage 6 — mechanism discussed only with evidence)

**M1 — `transactionid` 99.19%: structural, fully explained.** 2,664,312 missing iff event is view/addtocart; 22,457 present iff transaction (0 violations). Mechanism: by construction, not a statistical process — MCAR/MAR/MNAR language does not apply. Handling: grouping key only (§5/DQ1).

**M2 — As-of availability unknown 19.2%: informative, NOT MCAR.** Evidence against MCAR: unknown rate is 19.5% for views but 9.8% for carts / 11.1% for transactions; 36.8% for tail items vs 17.7% head; and 100% → ~10% across weeks (feed warm-up, `fig_asof_unknown_week.png`). So missingness correlates with the candidate target, popularity, and time — MCAR is rejected by the data. MAR reading (defensible): week + item-age explain most of it (early-window left-censoring: property feed starts after events; 49,815 event items — 21% — have no property rows at all). MNAR residue (not ruled out): tail items may lack rows *because* they are obscure/dead stock. Handling: explicit `unknown` state for the as-of flag (never impute 0/1 — unknown concentrates in views and would inject bias); add item-property-coverage indicator; warm-up weeks need care in temporal splits.

**M3 — `parentid` 1.5%: root markers.** 25/1,669 null = hierarchy roots. Not missing data at all. Handling: `is_root` boolean.

**M4 — EAV silence as signal.** An item lacking an `available` row at event time (M2) or lacking whole properties (21% of event items) is itself information about catalog coverage. Handling: coverage-count features (n properties known as-of) rather than row-dropping; dropping would delete 1 in 5 interacted items and bias toward the catalogued head.


In [ ]:
# 11a. Missingness — by-group rates from the measured artifact (evidence, not recounts)
import json
from IPython.display import Image, display

missing_groups = json.load(open("../reports/stage6_missing.json"))
print("as-of unknown overall %:", missing_groups["asof_unknown_overall_pct"])
print("by event:", missing_groups["asof_unknown_by_event_pct"])
print("by popularity:", missing_groups["asof_unknown_by_pop_pct"])
print("event items missing from property universe:",
      missing_groups["event_items_missing_from_properties"], "of", missing_groups["event_items"])
display(Image(filename="../reports/figures/fig_asof_unknown_week.png"))


## 12. Feature Relationships & Redundancy (completed Stage 6)

**Linear vs rank structure (why ranks won).** Pearson overstates every item-metric link vs Spearman by +0.04 to +0.21 (delta heatmap `fig_pearson_spearman_delta.png`; e.g. carts↔txs 0.806 vs 0.681, views↔txs 0.540 vs 0.326; full tables `stage6_corr_pearson.csv` / `stage5_corr_items.csv`). The gap is head-outlier leverage on least squares — with kurtosis in the hundreds, Pearson measures the head, Spearman measures the ordering. Decision: rank/quantile methods throughout; no linear model touches raw counts.

**Redundancy verdicts.** (1) Item views↔visitors: Spearman 0.976, Pearson 0.987 — drop one (keep views; visitors adds nothing at item grain). (2) Carts↔txs 0.681 — linked stages, keep both as separate stage labels, never as mutual features (a cart indicator predicting its own session's transaction is near-label). (3) Views↔txs 0.326 — volume ≠ conversion; both survive as distinct signals (popularity prior + conversion residual).

**Leakage re-audit (quantified).** `transactionid` (perfect label proxy), total session length (post-hoc, §7.3), naive latest-value joins (25.5% disagreement, §7.2) remain banned; carts-as-feature-for-buy is stage leakage; latest-category mapping is association-only. No new leakage vectors found in §§10–12.

**Transformation opportunities (from shape, not habit).** log1p/quantile for `790`-style magnitudes (skew 19.8) and count features (skew 10–270); EB smoothing (m=500) already validated for root rates; popularity deciles already validated as the working discretization. Correlation is never read causally anywhere in §§7–12.


In [ ]:
# 12a. Relationships — Pearson vs Spearman delta (measured tables)
import pandas as pd
from IPython.display import Image, display

spearman_table = pd.read_csv("../reports/stage5_corr_items.csv", index_col=0)
pearson_table = pd.read_csv("../reports/stage6_corr_pearson.csv", index_col=0)
print("Spearman:\n", spearman_table.round(2).to_string())
print("Pearson minus Spearman:\n", (pearson_table - spearman_table).round(2).to_string())
display(Image(filename="../reports/figures/fig_pearson_spearman_delta.png"))


## 13. Statistical Analysis (completed Stage 6 — four pre-registered tests, effects first)

Family-wise note: 4 tests on Q1/Q4/Q5 items → Bonferroni α = 0.0125. With n in the 10⁵–10⁶ range every null is rejected; the verdicts below therefore rest on effect sizes and CIs, and each p-value is reported once for completeness. Computations use stdlib distributions (`reports/stage6_c2.py`; scipy DLL blocked in this sandbox); helpers `wilson_interval` / `cramers_v_from_chi2` live in `src.analysis` and are unit-tested.

**T1 — Availability and tx rate (Q4).** H0: in-stock and out-of-stock as-of tx rates are equal; H1: they differ. Two-proportion z (n₁=99,468, n₀=62,152; normal approx valid at this n). Result: 1.360% vs 0.151%, Δ = 1.209 pp, 95% CI [1.131, 1.287], lift 8.99x, z = 25.1, p ≈ 5e-139. Interpretation: the gap is far beyond sampling noise and the CI excludes anything negligible — availability is the strongest documented associate; still associational (popularity confounding per §8.1), never causal.

**T2 — Item views: converting vs non-converting items (Q6).** H0: same view-count distribution; H1: different. Mann–Whitney U (medians 38 vs 3, IQRs disjoint — t-test assumptions untenable, ranks appropriate). Result: U ≈ 2.47e9, z ≈ 155, p ≈ 0, rank-biserial |r| = 0.838 (large). Interpretation: items that ever convert live in a different popularity stratum — consistent with the decile monotonicity, and a warning that unadjusted models will proxy popularity.

**T3 — Root branch × outcome independence (Q5).** H0: transaction incidence independent of root (top-10 + rest, event grain); H1: dependent. Chi-square, dof=10. Result: χ² = 2256, p ≈ 0, Cramér's V = 0.029 (tiny). Interpretation: the honest split verdict — branch dependence is real but *weak as a raw classifier* at event grain; its modeling value is hierarchical back-off for cold items (§7.5 smoothing), not direct discrimination. Reporting V prevents the p≈0 from overselling.

**T4 — Weekly tx homogeneity (Q1 split validity).** H0: equal tx proportions across the 19 full weeks; H1: some differ. Chi-square, dof=18. Result: χ² = 122.7, p ≈ 1e-17 — yet the range is only 7.17–9.66/1000 (±15% relative). Interpretation: textbook statistical-vs-practical significance — detectable wobble, no regime shift; temporal splits stand, no regime dummies needed.

**Rate anchors (Wilson 95%).** Event tx rate 0.815% [0.804, 0.826]; session buy rate 0.812% [0.798, 0.825] — the funnel denominator is measured to ±0.01 pp.


In [ ]:
# 13a. Stats — cross-check src helpers against the measured artifact (stdlib only)
import json
from src.analysis import cramers_v_from_chi2, wilson_interval

stats_results = json.load(open("../reports/stage6_stats.json"))
print("T1 lift:", stats_results["T1_availability_tx_rates"]["lift"],
      "| CI95 pp:", stats_results["T1_availability_tx_rates"]["ci95_pp"])
print("T2 |rank-biserial|:", abs(stats_results["T2_mwu_item_views"]["rank_biserial"]))
print("T3 V (artifact):", stats_results["T3_chi2_root_outcome"]["cramers_v"],
      "| src:", cramers_v_from_chi2(2256.0, 2756101, 11, 2))
print("T4 p:", stats_results["T4_chi2_weekly_homogeneity"]["p"], "(effect: range 7.17-9.66/1000)")
print("Wilson event tx% (src):", wilson_interval(22457, 2756101),
      "| artifact:", stats_results["wilson_cis"]["event_tx_rate_wilson95_pct"])


## 14. Potential Feature Engineering Opportunities (completed Stage 6 — proposals only, no pipeline)

| # | Feature family | Evidence it earns inclusion | Leakage / handling note |
|---|---|---|---|
| F1 | As-of `available` flag + `unknown` state | 9x tx lift, survives popularity split (§§7.2, 8.1) | Point-in-time only; `unknown` explicit (M2); naive join banned |
| F2 | Popularity-decile prior (views) | Monotonic 0.7→10.3 tx/1000 (§7.4); T2 | Compute from train-window only; tail slices in eval |
| F3 | Root back-off encoding (EB-smoothed) | 45x smoothed spread; V=0.029 warns against raw use (§§7.5, T3) | Latest-category caveat; smoothing m≈500 |
| F4 | Session-prefix dynamics (cart-so-far, events-so-far, time-since-last) | Length separates outcomes (§7.3) | Prefix-legal only; total length banned |
| F5 | log1p/quantile of count & `790`-style magnitudes | Skew 5–270; Pearson distortion (§12) | Fit mappings on train fold |
| F6 | Coverage indicators (n properties known as-of) | 21% items uncovered; unknown is informative (M2/M4) | Cheap, leak-free |
| F7 | Date-derived: week index, days-since-first-seen (item/visitor) | Stationary weeks (T4); warm-up decay (M2) | No absolute timestamps as magnitudes |
| F8 | Rare-code handling: frequency-gated hashing for EAV tail | 1,050-code tail (§6) | Thresholds from train; `283`-style lists need tokenizers |
| F9 | Missingness indicators (as-of unknown) | Unknown correlates with target (M2) | Indicator, never imputed 0/1 |

Explicitly deferred: text embeddings of multi-token values, sequential/deep session models, resampling policy — modeling-stage decisions. Dimensionality reduction is unnecessary: the engineered set is ~15 wide, not hundreds.


In [ ]:
# 14a. FE opportunities — decision checklist (documents scope; engineers nothing)
feature_checklist = [
    ("F1 as-of available + unknown", "approved", "§§7.2/8.1/M2"),
    ("F2 popularity-decile prior", "approved", "§7.4/T2"),
    ("F3 root back-off (EB m=500)", "approved with caveat", "§7.5/T3"),
    ("F4 session-prefix dynamics", "approved if prefix-legal", "§7.3/§8.3"),
    ("F5 log/quantile transforms", "approved", "§12"),
    ("F6 coverage indicators", "approved", "§11 M2/M4"),
    ("F7 week / days-since-first-seen", "approved", "T4/M2"),
    ("F8 rare-code hashing", "approved", "§6"),
    ("F9 missingness indicators", "approved", "§11 M2"),
]
for name, status, evidence in feature_checklist:
    print(f"{status:22s} {name:34s} ({evidence})")
print("\nDeferred to modeling: text embeddings, sequential models, resampling policy.")


## 15. ML Readiness Assessment (Stage 7 -- verdict: ready with conditions)

**Data quality: PASS WITH CONTROLS.** Missingness is mapped, not mysterious: `transactionid` structural (grouping key only), as-of `unknown` 19% with an explicit-state policy (M2), `parentid` roots flagged, zero invalid numerics, 460 duplicate rows quarantined with a sensitivity policy, outliers triaged to retain+transform (O1-O5). No silent cleaning anywhere; every issue has Observation -> Evidence -> Impact -> Action (Sections 5, 10, 11).

**Feature quality: CONDITIONAL PASS.** Approved: as-of availability+unknown (F1), popularity-decile prior (F2), EB-smoothed root back-off (F3), prefix-legal session dynamics (F4), log/quantile transforms (F5), coverage + missingness indicators (F6/F9), time features (F7), rare-code hashing (F8). Redundant: item-level visitors (rho=0.98 with views -- dropped). High-cardinality IDs (1.41M visitors, 235k items) are aggregation keys only, never raw features. Leakage gates enforced: `transactionid`, total session length, naive joins, and carts-as-feature-for-buy are banned with measured justification (25.5% naive-join disagreement, Sec 7.2).

**Target quality: VIABLE, EXTREME IMBALANCE.** Session-buy 0.812% [0.798, 0.825], event-tx 0.815% [0.804, 0.826] (Wilson 95 -- denominators known to +-0.01 pp). Label concerns contained: cart-stage dup inflation ~0.7%, latest-category misattribution flagged, 19% as-of-unknown handled by explicit state. The rate survives three grains and 19 stable weeks -- structural signal, not sampling artifact.

**Modeling considerations (binding constraints for training).** Scaling: rank/quantile/log -- never raw counts (Pearson distortion Sec 12). Encoding: EB smoothing (m=500) for roots, deciles for popularity, hashing for the EAV tail. Splitting: temporal (last weeks held out) + session-aware grouping (never split a session across folds); stratify by outcome and week. Validation: PR-AUC primary, recall@precision operating points; tail-slice and cold-start reporting mandatory. Cross-validation: time-series folds (rolling origin), never shuffled K-fold. Residual risks: dictionary-free property semantics (only `available`/`categoryid` trusted), MNAR residue in tail missingness, warm-up-week bias.

**Verdict: READY WITH CONDITIONS** -- the six gates below must hold before the first training run. No model is trained here: the EDA's job was to make training trustworthy, and these gates are its output.


In [ ]:
# 15a. Readiness -- gate checklist (all gates trace to measured sections)
readiness_gates = [
    ("As-of property pipeline (naive join 25.5% wrong)", "required", "Secs 7.2/8.1"),
    ("Banned list: transactionid, session length, naive joins", "required", "Secs 7.3/12"),
    ("Temporal + session-aware splits, stratify outcome x week", "required", "Sec 9/T4"),
    ("PR-AUC + tail-slice reporting; accuracy banned", "required", "Secs 9/7.4"),
    ("Explicit unknown state for as-of features", "required", "Sec 11 M2"),
    ("Rank/log transforms; drop item-visitors", "required", "Sec 12"),
]
for gate, status, evidence in readiness_gates:
    print("[" + status.upper() + "] " + gate + " (" + evidence + ")")
print("Verdict: READY WITH CONDITIONS -- no training until all six gates hold.")


## 16. Key Findings (Stage 7 -- insight only, no generic counts)

1. **Finding:** Conversion is structurally ~0.8% at every grain. **Evidence:** event 0.815%, session-buy 0.812%, visitor-buyer 0.833%; Wilson CIs +-0.01 pp (Secs 7.6, 9, T4). **Why it matters:** fixes sampling budgets (~8 tx/1000 views) and bans accuracy as a metric.
2. **Finding:** In-stock state associates with ~9x conversion and survives the popularity split. **Evidence:** 1.36% vs 0.15% tx, Delta CI [1.13, 1.29] pp, z=25.1; head 0.17->1.36%, tail 0.01->1.23% (Secs 7.2, 8.1, T1). **Why it matters:** justifies the as-of availability feature family -- and the as-of pipeline (25.5% naive disagreement).
3. **Finding:** Sessionization is gap-robust; sessions are the right decision unit. **Evidence:** 1.73-1.81M sessions across 15/30/60 min; 27.2% of cart sessions convert; buy sessions average 9.3 events (Secs 7.1, 7.3, 9). **Why it matters:** deterministic session ids + session-buy label; total length banned as post-hoc leakage.
4. **Finding:** The catalog head converts better at both funnel stages. **Evidence:** monotonic deciles 0.7->10.3 tx/1000 views; converting items' median 38 views vs 3 (rank-biserial 0.84) (Sec 7.4, T2). **Why it matters:** popularity priors help but evaluation must include tail slices.
5. **Finding:** Category branch carries back-off signal, not discrimination. **Evidence:** 45x smoothed spread yet Cramer's V=0.029 (Sec 7.5, T3). **Why it matters:** hierarchical encoding for cold start, not a raw classifier -- prevents overselling a p~0.
6. **Finding:** Missingness is informative, and the feed has a warm-up. **Evidence:** unknown 2x higher in views than carts, 100%->10% across weeks, 21% of event items uncovered (Sec 11 M2). **Why it matters:** explicit unknown states; warm-up care in splits.
7. **Finding:** Extremes are genuine -- transform, never delete. **Evidence:** hyperactive buyers convert above base rate; 31-item baskets are distinct SKUs; fences flag up to 22% (Sec 10). **Why it matters:** rank/log representations instead of deletion caps.
8. **Finding:** Linear methods misread this data. **Evidence:** Pearson overstates every link by up to +0.21; weekly wobble is significant (p~1e-17) yet practically +-15% (Sec 12, T4). **Why it matters:** rank statistics, effect sizes over p-values, no regime dummies.


In [ ]:
# 16a. Key findings -- the three figures that carry the story (measured artifacts)
from IPython.display import Image, display

for key_fig in ["fig_funnel.png", "fig_asof_available.png", "fig_popularity_conversion.png"]:
    display(Image(filename="../reports/figures/" + key_fig))
print("8 findings, each Finding -> Evidence -> Why it matters; see Sec 16 markdown.")


## 17. Recommendations (Stage 7 -- every item traces to analysis)

**Data recommendations.** (1) Publish the warm-up cutoff: exclude or down-weight week 1 in training (100% as-of-unknown, M2). (2) Keep the 460-duplicate quarantine with with/without sensitivity in the pipeline (DQ2). (3) Acquire or decode the property dictionary -- only `available`/`categoryid` are trusted; `790` units and multi-token codes are the top semantic risk (DQ5/O4). (4) Monitor marathon sessions and `visitorid=0` as bot/abuse signals without deleting them (O1/O3/DQ3).

**Modeling recommendations.** (1) Session-buy framing with prefix-legal features F1-F9 (Sec 14). (2) Temporal last-weeks holdout + session-aware grouping + outcome/week stratification; rolling-origin CV, never shuffled K-fold (Secs 9, 15). (3) PR-AUC primary with recall@precision operating points; mandatory head/tail/cold-start slices (Secs 7.4, 9). (4) Enforce the six readiness gates (Sec 15) as pipeline assertions before any training run -- especially the as-of join and banned-feature list.

**Business recommendations (associational -- test before acting).** (1) Stockout coincidence is the largest documented associate of non-conversion (~9x gap): audit whether high-intent items go out of stock mid-funnel rather than assuming demand is absent (Q4). (2) Cart sessions convert at 27% -- cart-stage recovery targets the highest-yield segment, not generic traffic (Q1). (3) Branch 1224 converts at ~2x the global rate while tail branches sit near zero: differentiate follow-ups by branch with shrinkage-aware reporting (Q5). (4) The catalog tail (21% of interacted items uncovered by properties) is a data-coverage investment with direct modeling payoff (M2/M4).


In [ ]:
# 17a. Recommendations -- traceability check (each item cites its evidence section)
recommendation_trace = {
    "data": ["M2 warm-up cutoff", "DQ2 dup quarantine", "DQ5/O4 dictionary gap", "O1/O3/DQ3 monitoring"],
    "modeling": ["F1-F9 session-buy framing (Sec 14)", "temporal+session splits (Sec 9)",
                 "PR-AUC + slices (Sec 9)", "six gates (Sec 15)"],
    "business": ["stockout audit (Q4)", "cart recovery 27% (Q1)", "branch 1224 vs tail (Q5)",
                 "tail coverage (M2/M4)"],
}
for track, items in recommendation_trace.items():
    print(track + ": " + str(len(items)) + " recommendations, all evidence-traced")


## 18. Conclusion (Stage 7)

This EDA converted a 2.76M-event / 20.3M-row / three-table log with no dictionary, no target, and no sessions into a modellable problem: session-buy prediction at a structural ~0.8% base rate, with an as-of availability flag, popularity-decile priors, and EB-smoothed hierarchical back-off as the evidence-backed feature core -- gated by six binding readiness conditions. Proven: conditional signal far above base rates, stationary weeks, gap-robust sessions. Still unknown: property semantics beyond two codes, MNAR residue in tail missingness, intent behind marathon sessions. The single next step before training: implement the six gates as pipeline assertions and re-verify the as-of disagreement rate on the training window.


In [ ]:
# 18a. Provenance -- what this notebook rests on (artifacts, not claims)
from pathlib import Path

artifact_files = sorted(Path("../reports").glob("stage*.json"))
print(str(len(artifact_files)) + " measurement artifacts + 18 figures under reports/figures/")
print("Tests: python -m pytest tests/ -q (stdlib-only; scipy pinned for standard envs)")
print("Single next step: implement the six Sec-15 gates as pipeline assertions.")


In [ ]:
# Final check: imports resolve and raw inventory is non-empty (no heavy parsing)
final_raw_files
assert final_raw_files, "data/raw appears empty â€” see data/README.md"
print(f"final OK: {len(final_raw_files)} raw files inventoried")
